### Cell 1 — Setup

In [0]:
# ============================================================
#  04_ML_CLUSTERING — TOURGO ML PIPELINE
#  Day 5: Feature Engineering cho K-Means
#  Tạo ra: ml_customer_features (7 features)
# ============================================================

from pyspark.sql.functions import (
    col, count, sum as _sum, avg, countDistinct,
    when, datediff, to_date, lit,
    round as _round, coalesce
)
from pyspark.sql.types import DoubleType, IntegerType
from datetime import date

spark.sql("USE tourgo")

# Ngày export — dùng để tính days_active
# Thay bằng ngày thật khi chạy
EXPORT_DATE = str(date.today())

print("   Setup xong")
print(f"  EXPORT_DATE = {EXPORT_DATE}")
print("   Sẽ tạo: ml_customer_features")

   Setup xong
  EXPORT_DATE = 2026-06-02
   Sẽ tạo: ml_customer_features


### Cell 2 — Đọc Silver tables

In [0]:
# Đọc các Silver tables cần dùng
df_bookings = spark.read.table("silver_bookings")
df_revenues  = spark.read.table("silver_revenues")
df_reviews   = spark.read.table("silver_reviews")
df_users     = spark.read.table("silver_users") \
                    .filter(col("role") == "CUSTOMER")

print("    Silver tables loaded:")
print(f"   bookings : {df_bookings.count():,}")
print(f"   revenues : {df_revenues.count():,}")
print(f"   reviews  : {df_reviews.count():,}")
print(f"   customers: {df_users.count():,}")

    Silver tables loaded:
   bookings : 358
   revenues : 114
   reviews  : 109
   customers: 250


### Cell 3 — Feature 1–4: từ bookings

In [0]:
# ── Features từ bảng bookings ──────────────────────────────
# Feature 1: total_bookings   — mức độ hoạt động
# Feature 2: confirmed_bookings — tỷ lệ hoàn thành
# Feature 3: cancelled_bookings — để tính cancellation_rate
# Feature 4: unique_tours     — sự đa dạng hành vi
# Feature 5: cancellation_rate — độ tin cậy
from pyspark.sql.functions import col, upper, count, when, round, sum

print("  Tính features từ bookings...")

# Cell 3 — sửa filter status bookings
df_booking_features = df_bookings \
    .groupBy("user_id") \
    .agg(
        count("id").alias("total_bookings"),
        count(when(upper(col("status")) == "CONFIRMED", 1))  # <- CONFIRMED
            .alias("confirmed_bookings"),
        count(when(upper(col("status")) == "CANCELLED", 1))  # <- CANCELLED
            .alias("cancelled_bookings"),
        countDistinct("tour_id").alias("unique_tours")
    ) \
    .withColumn(
        "cancellation_rate",
        _round(
            when(col("total_bookings") > 0,
                 col("cancelled_bookings").cast(DoubleType()) /
                 col("total_bookings").cast(DoubleType())
            ).otherwise(0.0),
            3
        )
    )

print(f" Booking features: {df_booking_features.count()} users có booking")
df_booking_features.show(5, truncate=False)

  Tính features từ bookings...
 Booking features: 241 users có booking
+-------+--------------+------------------+------------------+------------+-----------------+
|user_id|total_bookings|confirmed_bookings|cancelled_bookings|unique_tours|cancellation_rate|
+-------+--------------+------------------+------------------+------------+-----------------+
|230    |2             |2                 |0                 |2           |0.0              |
|231    |2             |2                 |0                 |2           |0.0              |
|232    |2             |1                 |1                 |2           |0.5              |
|276    |2             |1                 |1                 |2           |0.5              |
|317    |2             |2                 |0                 |1           |0.0              |
+-------+--------------+------------------+------------------+------------+-----------------+
only showing top 5 rows


### Cell 4 — Feature: total_spent từ revenues

In [0]:
# ── Feature total_spent từ revenues ────────────────────────
# revenues.payment_id → bookings.id → user_id

print("  Tính total_spent từ revenues...")

df_spending = df_revenues \
    .join(
        df_bookings.select(
            col("id").alias("booking_id"),
            col("user_id")
        ),
        col("payment_id") == col("booking_id"),
        "left"
    ) \
    .groupBy("user_id") \
    .agg(
        _round(_sum("total_amount"), 0).alias("total_spent")
    )

print(f" Spending features: {df_spending.count()} users có chi tiêu")
df_spending.show(5, truncate=False)

  Tính total_spent từ revenues...
 Spending features: 99 users có chi tiêu
+-------+-----------+
|user_id|total_spent|
+-------+-----------+
|232    |7500000.0  |
|347    |1500000.0  |
|271    |6000000.0  |
|221    |2000000.0  |
|325    |4500000.0  |
+-------+-----------+
only showing top 5 rows


### Cell 5 — Feature: avg_rating_given từ reviews

In [0]:
# ── Feature avg_rating_given từ reviews ────────────────────

print("  Tính avg_rating_given từ reviews...")

df_rating_features = df_reviews \
    .groupBy("user_id") \
    .agg(
        _round(avg("rating"), 2).alias("avg_rating_given"),
        count("id").alias("review_count")
    )

print(f" Rating features: {df_rating_features.count()} users đã review")
df_rating_features.show(5, truncate=False)

  Tính avg_rating_given từ reviews...
 Rating features: 95 users đã review
+-------+----------------+------------+
|user_id|avg_rating_given|review_count|
+-------+----------------+------------+
|322    |5.0             |1           |
|441    |5.0             |2           |
|402    |5.0             |1           |
|318    |4.0             |1           |
|315    |5.0             |1           |
+-------+----------------+------------+
only showing top 5 rows


### Cell 6 — Feature: days_active từ users

In [0]:
# ── Feature days_active từ users ───────────────────────────
# Tính số ngày kể từ khi tham gia → chia 30 = số tháng

print("Tính days_active từ users...")

df_user_age = df_users \
    .withColumn(
        "days_active",
        datediff(
            to_date(lit(EXPORT_DATE)),
            to_date(col("date_joined"))
        ).cast(IntegerType())
    ) \
    .select(
        col("id").alias("user_id"),
        col("days_active")
    )

print(f"User age features: {df_user_age.count()} customers")
df_user_age.show(5, truncate=False)

Tính days_active từ users...
User age features: 250 customers
+-------+-----------+
|user_id|days_active|
+-------+-----------+
|221    |80         |
|224    |75         |
|232    |69         |
|248    |75         |
|262    |82         |
+-------+-----------+
only showing top 5 rows


### Cell 7 — JOIN tất cả features

In [0]:
# ── JOIN tất cả features thành 1 bảng ──────────────────────
print("  Joining all features...")

df_features = df_users.select(col("id").alias("user_id")) \
    .join(df_booking_features, "user_id", "left") \
    .join(df_spending,         "user_id", "left") \
    .join(df_rating_features,  "user_id", "left") \
    .join(df_user_age,         "user_id", "left") \
    .fillna({
        "total_bookings":    0,
        "confirmed_bookings":0,
        "cancelled_bookings":0,
        "unique_tours":      0,
        "cancellation_rate": 0.0,
        "total_spent":       0.0,
        "avg_rating_given":  0.0,
        "review_count":      0,
        "days_active":       0
    })

# Lưu feature table
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_customer_features")

total = df_features.count()
with_bookings = df_features.filter(col("total_bookings") > 0).count()

print(f"   ml_customer_features: {total:,} customers")
print(f"   Có bookings: {with_bookings:,} ({with_bookings/total*100:.1f}%)")
print(f"   Không có bookings: {total - with_bookings:,}")
df_features.show(5, truncate=False)

  Joining all features...
   ml_customer_features: 250 customers
   Có bookings: 241 (96.4%)
   Không có bookings: 9
+-------+--------------+------------------+------------------+------------+-----------------+-----------+----------------+------------+-----------+
|user_id|total_bookings|confirmed_bookings|cancelled_bookings|unique_tours|cancellation_rate|total_spent|avg_rating_given|review_count|days_active|
+-------+--------------+------------------+------------------+------------+-----------------+-----------+----------------+------------+-----------+
|221    |1             |1                 |0                 |1           |0.0              |2000000.0  |0.0             |0           |80         |
|224    |1             |1                 |0                 |1           |0.0              |0.0        |4.0             |1           |75         |
|232    |2             |1                 |1                 |2           |0.5              |7500000.0  |0.0             |0           |69      

### Cell 8 — Distribution Check (gửi cho [A] Hà)

In [0]:
# ============================================================
#  DISTRIBUTION CHECK — gửi kết quả này cho [A] Hà validate
# ============================================================

df_feat = spark.read.table("ml_customer_features")
total = df_feat.count()

print("=" * 55)
print(" FEATURE DISTRIBUTION CHECK — GỬI CHO [A] HÀ")
print("=" * 55)

checks = {
    "Users tổng":                total,
    "Users có booking":          df_feat.filter(col("total_bookings") > 0).count(),
    "Users booking > 5":         df_feat.filter(col("total_bookings") > 5).count(),
    "Users đã chi tiêu":         df_feat.filter(col("total_spent") > 0).count(),
    "Users đã review":           df_feat.filter(col("review_count") > 0).count(),
    "Users confirmed > 0":       df_feat.filter(col("confirmed_bookings") > 0).count(),
    "Users cancellation > 50%":  df_feat.filter(col("cancellation_rate") > 0.5).count(),
}

for label, val in checks.items():
    pct = f"({val/total*100:.1f}%)" if total > 0 else ""
    print(f"  {label:<30}: {val:>5,} {pct}")

print("=" * 55)

# Thống kê mô tả 7 features
print("\n DESCRIBE — 7 FEATURES:")
df_feat.select(
    "total_bookings", "confirmed_bookings", "total_spent",
    "cancellation_rate", "avg_rating_given", "unique_tours", "days_active"
).describe().show(truncate=False)

# Quyết định augment
pct_no_booking = (total - checks["Users có booking"]) / total * 100
print("=" * 55)
if pct_no_booking > 80:
    print(f"--ERROR--  {pct_no_booking:.1f}% users không có booking")
    print("-> CẦN AUGMENT DATA — báo [D] Khang chạy seed_fake_data.py")
    print("   Target: ít nhất 200 bookings để K-Means có ý nghĩa")
else:
    print(f"--OK-- {100 - pct_no_booking:.1f}% users có booking — đủ data")
    print("-> KHÔNG cần augment, tiến hành Day 6 (Train K-Means)")
print("=" * 55)

 FEATURE DISTRIBUTION CHECK — GỬI CHO [A] HÀ
  Users tổng                    :   250 (100.0%)
  Users có booking              :   241 (96.4%)
  Users booking > 5             :     0 (0.0%)
  Users đã chi tiêu             :    99 (39.6%)
  Users đã review               :    95 (38.0%)
  Users confirmed > 0           :   227 (90.8%)
  Users cancellation > 50%      :     8 (3.2%)

 DESCRIBE — 7 FEATURES:
+-------+------------------+------------------+-----------------+-------------------+-----------------+------------------+-----------------+
|summary|total_bookings    |confirmed_bookings|total_spent      |cancellation_rate  |avg_rating_given |unique_tours      |days_active      |
+-------+------------------+------------------+-----------------+-------------------+-----------------+------------------+-----------------+
|count  |250               |250               |250              |250                |250              |250               |250              |
|mean   |1.432             |1.2

### Cell 9 — Preview feature table (để chụp screenshot)

In [0]:
# ============================================================
#  PREVIEW — Top 10 customers theo total_spent
#  Chụp screenshot cell này cho docs/screenshots/day5/
# ============================================================

print(" TOP 10 CUSTOMERS (by total_spent):")
spark.read.table("ml_customer_features") \
    .orderBy(col("total_spent").desc()) \
    .select(
        "user_id",
        "total_bookings",
        "confirmed_bookings",
        "total_spent",
        "cancellation_rate",
        "avg_rating_given",
        "unique_tours",
        "days_active"
    ) \
    .limit(10) \
    .show(truncate=False)

print("\n Day 5 hoàn tất!")
print("   -> Gửi output Cell 8 cho [A] Hà")
print("   -> Chờ [A] Hà xác nhận trước khi sang Day 6")

 TOP 10 CUSTOMERS (by total_spent):
+-------+--------------+------------------+-----------+-----------------+----------------+------------+-----------+
|user_id|total_bookings|confirmed_bookings|total_spent|cancellation_rate|avg_rating_given|unique_tours|days_active|
+-------+--------------+------------------+-----------+-----------------+----------------+------------+-----------+
|212    |2             |2                 |2.1E7      |0.0              |3.5             |2           |71         |
|410    |2             |1                 |2.06E7     |0.0              |0.0             |2           |81         |
|301    |2             |2                 |1.88E7     |0.0              |0.0             |2           |72         |
|353    |2             |2                 |1.48E7     |0.0              |5.0             |2           |86         |
|252    |1             |0                 |1.4E7      |1.0              |0.0             |1           |89         |
|228    |1             |1           

# HÀ XÁC NHẬN TRƯỚC KHI SANG DAY 6 NHÉ T_T

# Cell 10 - Setup & Load

In [0]:
# ============================================================
#  RESTART PYTHON — Xóa toàn bộ ML cache và session state
# ============================================================
# Chạy cell này KHI gặp lỗi ML_CACHE_SIZE_OVERFLOW_EXCEPTION
# SAU KHI restart, chạy lại Cell 21 (Setup & Load) trước

dbutils.library.restartPython()

In [0]:
# ============================================================
#  DAY 6 — K-Means Train & Interpret
#  Tiếp tục trong 04_ml_clustering.ipynb (sau Cell 9 của Day 5)
# ============================================================

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, count, avg, round as _round
import mlflow

spark.sql("USE tourgo")

df_feat = spark.read.table("ml_customer_features")

FEATURE_COLS = [
    "total_bookings", "confirmed_bookings", "total_spent",
    "cancellation_rate", "avg_rating_given", "unique_tours", "days_active"
]

# Verify data trước khi train
total = df_feat.count()
print(f"   ml_customer_features: {total:,} customers")
print(f"   Features sử dụng: {FEATURE_COLS}")

# Kiểm tra không có NULL trong feature columns
null_check = df_feat.select([
    count(col(c)).alias(c) for c in FEATURE_COLS
]).collect()[0]
print("\n Non-null counts per feature:")
for f in FEATURE_COLS:
    print(f"   {f:<25}: {null_check[f]:,} / {total:,}")

   ml_customer_features: 250 customers
   Features sử dụng: ['total_bookings', 'confirmed_bookings', 'total_spent', 'cancellation_rate', 'avg_rating_given', 'unique_tours', 'days_active']

 Non-null counts per feature:
   total_bookings           : 250 / 250
   confirmed_bookings       : 250 / 250
   total_spent              : 250 / 250
   cancellation_rate        : 250 / 250
   avg_rating_given         : 250 / 250
   unique_tours             : 250 / 250
   days_active              : 250 / 250


### Cell 11 - Cell 11 — Elbow Method K=2..8

In [0]:
# ─── Clear ML Cache trước khi chạy Elbow Method ────────────
# Chạy cell này NẾU gặp lỗi ML_CACHE_SIZE_OVERFLOW_EXCEPTION

import gc

# Xóa tất cả biến model/pipeline có thể còn trong session
try:
    del model, pipeline, kmeans, assembler, scaler, preds
except NameError:
    pass

gc.collect()
print("✓ Đã clear ML cache. Có thể chạy Cell 23 (Elbow Method) ngay bây giờ.")

✓ Đã clear ML cache. Có thể chạy Cell 23 (Elbow Method) ngay bây giờ.


In [0]:
# ── Elbow Method ────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="raw_features",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)
evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette"
)

silhouette_scores = {}
inertia_scores    = {}

mlflow.set_experiment("/Shared/tourgo-ml-experiments")

with mlflow.start_run(run_name="kmeans_elbow_method"):
    mlflow.log_param("features", FEATURE_COLS)
    mlflow.log_param("scaler",   "StandardScaler")
    mlflow.log_param("k_range",  "2-8")
    mlflow.log_param("dataset_size", total)

    for k in range(2, 9):
        kmeans   = KMeans(k=k, seed=42, maxIter=100)
        pipeline = Pipeline(stages=[assembler, scaler, kmeans])
        model    = pipeline.fit(df_feat)
        preds    = model.transform(df_feat)

        silhouette = evaluator.evaluate(preds)
        inertia    = model.stages[-1].summary.trainingCost

        silhouette_scores[k] = silhouette
        inertia_scores[k]    = inertia

        mlflow.log_metric(f"silhouette_k{k}", silhouette)
        mlflow.log_metric(f"inertia_k{k}",    inertia)
        print(f"  K={k} | Silhouette: {silhouette:.4f} | Inertia: {inertia:,.2f}")

# In bảng kết quả gửi [A] Hà
print("\n" + "="*58)
print("-> ELBOW METHOD — GỬI CHO [A] HÀ CHỌN BEST_K")
print("="*58)
print(f"  {'K':>3} | {'Silhouette':>12} | {'Inertia':>14} | Gợi ý")
print("  " + "-"*50)
best_k_sil = max(silhouette_scores, key=silhouette_scores.get)
for k in range(2, 9):
    tag = " <- silhouette cao nhất" if k == best_k_sil else ""
    print(f"  {k:>3} | {silhouette_scores[k]:>12.4f} | "
          f"{inertia_scores[k]:>14,.2f}{tag}")
print("="*58)
print(f"\n-> Gửi bảng này cho [A] Hà")
print(f"-> Chờ Hà xác nhận BEST_K rồi mới chạy Cell tiếp theo")

  K=2 | Silhouette: 0.5301 | Inertia: 1,122.34
  K=3 | Silhouette: 0.4317 | Inertia: 998.46
  K=4 | Silhouette: 0.4109 | Inertia: 926.01
  K=5 | Silhouette: 0.5667 | Inertia: 745.92
  K=6 | Silhouette: 0.4701 | Inertia: 610.07
  K=7 | Silhouette: 0.5241 | Inertia: 616.87
  K=8 | Silhouette: 0.5192 | Inertia: 607.86

-> ELBOW METHOD — GỬI CHO [A] HÀ CHỌN BEST_K
    K |   Silhouette |        Inertia | Gợi ý
  --------------------------------------------------
    2 |       0.5301 |       1,122.34
    3 |       0.4317 |         998.46
    4 |       0.4109 |         926.01
    5 |       0.5667 |         745.92 <- silhouette cao nhất
    6 |       0.4701 |         610.07
    7 |       0.5241 |         616.87
    8 |       0.5192 |         607.86

-> Gửi bảng này cho [A] Hà
-> Chờ Hà xác nhận BEST_K rồi mới chạy Cell tiếp theo


### Cell 12 — Train Final Model

In [0]:
# ── Train Final Model ───────────────────────────────────────
# ĐỢI [A] HÀ XÁC NHẬN BEST_K TRƯỚC KHI CHẠY
# Đổi giá trị BEST_K bên dưới theo yêu cầu của Hà

BEST_K = 4  # [A] Hà xác nhận — đổi nếu cần

print(f"-> Training final K-Means với K={BEST_K}...")

with mlflow.start_run(run_name=f"kmeans_final_k{BEST_K}"):
    kmeans_final   = KMeans(k=BEST_K, seed=42, maxIter=200)
    pipeline_final = Pipeline(stages=[assembler, scaler, kmeans_final])
    model_final    = pipeline_final.fit(df_feat)
    df_clustered   = model_final.transform(df_feat)

    final_silhouette = evaluator.evaluate(df_clustered)
    mlflow.log_param("k",                 BEST_K)
    mlflow.log_param("maxIter",           200)
    mlflow.log_param("seed",              42)
    mlflow.log_metric("final_silhouette", final_silhouette)

    print(f"==> K={BEST_K} | Silhouette: {final_silhouette:.4f}")
    if final_silhouette >= 0.5:
        print("   -> Phân cụm tốt (≥ 0.5)")
    elif final_silhouette >= 0.3:
        print("   -> Phân cụm chấp nhận được (0.3–0.5)")
    else:
        print("   -> Silhouette thấp (< 0.3) — bình thường khi data ít")
        print("     Vẫn dùng được cho demo, ghi chú trong slide")

# Lưu vào managed table
df_clustered \
    .select("user_id", "prediction", *FEATURE_COLS) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_customer_segments")

seg_count = spark.read.table("ml_customer_segments").count()
print(f"=> ml_customer_segments: {seg_count:,} records")

-> Training final K-Means với K=4...
==> K=4 | Silhouette: 0.4109
   -> Phân cụm chấp nhận được (0.3–0.5)
=> ml_customer_segments: 250 records


### Cell 13 — Cluster Profile (gửi [A] Hà đặt tên)

In [0]:
# ── Cluster Profile ─────────────────────────────────────────
df_cluster_profile = df_clustered \
    .groupBy("prediction") \
    .agg(
        count("user_id").alias("cluster_size"),
        _round(avg("total_bookings"),    2).alias("avg_bookings"),
        _round(avg("confirmed_bookings"),2).alias("avg_confirmed"),
        _round(avg("total_spent"),       0).alias("avg_spent"),
        _round(avg("cancellation_rate"), 3).alias("avg_cancel_rate"),
        _round(avg("avg_rating_given"),  2).alias("avg_rating"),
        _round(avg("unique_tours"),      2).alias("avg_unique_tours"),
        _round(avg("days_active"),       0).alias("avg_days_active")
    ) \
    .orderBy("prediction")

print("="*65)
print("-> CLUSTER PROFILE — GỬI CHO [A] HÀ ĐẶT TÊN CỤM")
print("="*65)
df_cluster_profile.show(truncate=False)

# Gợi ý tên dựa trên profile
print("-> GỢI Ý ĐẶT TÊN (dựa trên avg_spent và cancel_rate):")
for row in df_cluster_profile.collect():
    pct = row["cluster_size"] / total * 100
    hint = ""
    if row["avg_spent"] and row["avg_spent"] > 0:
        if float(row["avg_spent"]) > 5000000 and float(row["avg_cancel_rate"]) < 0.2:
            hint = "-> Khả năng: Khách VIP"
        elif float(row["avg_bookings"]) > 2 and float(row["avg_cancel_rate"]) < 0.3:
            hint = "-> Khả năng: Khách Tiềm Năng"
        elif float(row["avg_cancel_rate"]) > 0.5:
            hint = "-> Khả năng: Khách Không Ổn Định"
        else:
            hint = "-> Khả năng: Khách Ngủ Đông"
    print(f"  Cluster {row['prediction']}: "
          f"{row['cluster_size']:,} users ({pct:.1f}%) | "
          f"avg_spent={row['avg_spent']:,.0f} VNĐ | "
          f"cancel={row['avg_cancel_rate']:.1%} {hint}")

print("\n-> Gửi profile này cho [A] Hà xác nhận tên chính thức")
print("-> Gửi cho [D] Khánh để viết customer_segments_report.md")

-> CLUSTER PROFILE — GỬI CHO [A] HÀ ĐẶT TÊN CỤM
+----------+------------+------------+-------------+---------+---------------+----------+----------------+---------------+
|prediction|cluster_size|avg_bookings|avg_confirmed|avg_spent|avg_cancel_rate|avg_rating|avg_unique_tours|avg_days_active|
+----------+------------+------------+-------------+---------+---------------+----------+----------------+---------------+
|0         |57          |2.0         |1.89         |5522807.0|0.018          |4.18      |2.0             |78.0           |
|1         |9           |0.0         |0.0          |0.0      |0.0            |0.0       |0.0             |79.0           |
|2         |60          |2.0         |1.67         |1958333.0|0.117          |0.39      |1.98            |81.0           |
|3         |124         |1.0         |0.9          |1763710.0|0.056          |1.27      |1.0             |82.0           |
+----------+------------+------------+-------------+---------+---------------+----------+--

### Cell 14 — Tour Recommendations per Cluster

In [0]:
# ── Tour Recommendation dựa trên Cluster ───────────────────
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, upper as _upper

df_segments   = spark.read.table("ml_customer_segments")
df_bookings_s = spark.read.table("silver_bookings")
df_tours_s    = spark.read.table("silver_tours")
df_reviews_s  = spark.read.table("silver_reviews")

# Rating tổng hợp từng tour
df_tour_rating = df_reviews_s \
    .groupBy("tour_id") \
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        count("id").alias("review_count")
    )

# Tour được đặt nhiều trong mỗi cluster
# schema v3.1: bookings.status = CONFIRMED (uppercase)
df_cluster_tours = df_segments \
    .join(
        df_bookings_s.filter(_upper(col("status")) == "CONFIRMED"),
        "user_id",
        "inner"
    ) \
    .groupBy("prediction", "tour_id") \
    .agg(count("id").alias("bookings_by_cluster")) \
    .join(df_tour_rating, "tour_id", "left") \
    .join(
        df_tours_s.select(
            col("id").alias("t_id"),
            col("title"),
            col("price"),
            col("category_names")
        ),
        col("tour_id") == col("t_id"),
        "left"
    ) \
    .drop("t_id") \
    .filter(col("avg_rating") >= 3.5) \
    .withColumn(
        "score",
        col("bookings_by_cluster") * 0.6 +
        col("avg_rating") * 0.4 * 10000
    )

# Top 3 tours mỗi cluster
window_cluster = Window \
    .partitionBy("prediction") \
    .orderBy(col("score").desc())

df_recommendations = df_cluster_tours \
    .withColumn("rank", rank().over(window_cluster)) \
    .filter(col("rank") <= 3) \
    .select(
        "prediction", "rank", "title",
        "price", "avg_rating", "category_names",
        "bookings_by_cluster"
    ) \
    .orderBy("prediction", "rank")

df_recommendations.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_tour_recommendations")

print("-=> ml_tour_recommendations:")
spark.read.table("ml_tour_recommendations").show(truncate=False)

-=> ml_tour_recommendations:
+----------+----+---------------------------------+---------+----------+-----------------------------+-------------------+
|prediction|rank|title                            |price    |avg_rating|category_names               |bookings_by_cluster|
+----------+----+---------------------------------+---------+----------+-----------------------------+-------------------+
|0         |1   |Hội An Phố Cổ Huyền Ảo v18       |2000000.0|5.0       |Núi,Khám phá                 |2                  |
|0         |1   |Đà Nẵng - Hội An Nghỉ Dưỡng v6   |3000000.0|5.0       |Biển,Nghỉ dưỡng              |2                  |
|0         |1   |Huế Cố Đô Khám Phá Di Sản v21    |1700000.0|5.0       |Biển,Ẩm thực                 |2                  |
|0         |1   |Hội An Phố Cổ Huyền Ảo v30       |2000000.0|5.0       |Biển                         |2                  |
|0         |1   |Hội An Phố Cổ Huyền Ảo v23       |2000000.0|5.0       |Khám phá                     |2       

# Bị Tràn Bộ Nhớ Khi Đến Cell Code 15 - Cần chạy đoạn cell code dưới đây

In [0]:
# Giải phóng các biến model Python nếu còn giữ reference
import gc
try:
    del model
    del pipeline
    del kmeans
except NameError:
    pass

# Gọi ép buộc bộ thu gom rác của Python hoạt động
gc.collect()

# Cực kỳ quan trọng: Khởi động lại Spark Connect ML Cache (nếu Databricks yêu cầu)
# Bằng cách dọn sạch reference, Spark Connect sẽ tự giải phóng vùng đệm 1GB thô.

58485

### Cell 15 — PCA Visualization

In [0]:
# Cell 15 — Thay toàn bộ bằng scikit-learn PCA
# Không dùng Spark ML → không tốn ML cache

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.decomposition import PCA as SkPCA
import gc

print("  PCA bằng scikit-learn (tránh Spark ML cache overflow)...")

# Lấy features từ managed table thay vì refit
df_feat_pd = spark.read.table("ml_customer_features") \
    .select(*FEATURE_COLS, "user_id") \
    .toPandas()

df_seg_pd = spark.read.table("ml_customer_segments") \
    .select("user_id", "prediction") \
    .toPandas()

# Scale + PCA bằng sklearn
X = df_feat_pd[FEATURE_COLS].fillna(0).values
scaler_sk = SkScaler()
X_scaled  = scaler_sk.fit_transform(X)

pca_sk    = SkPCA(n_components=2, random_state=42)
X_pca     = pca_sk.fit_transform(X_scaled)

# Ghép kết quả
df_pca_pd = pd.DataFrame({
    "user_id":        df_feat_pd["user_id"].values,
    "pc1":            X_pca[:, 0],
    "pc2":            X_pca[:, 1],
})
df_viz_pd = df_pca_pd.merge(df_seg_pd, on="user_id", how="left")
df_viz_pd = df_viz_pd.merge(
    df_feat_pd[["user_id","total_spent","total_bookings"]],
    on="user_id", how="left"
)

# Lưu vào Delta managed table
df_viz = spark.createDataFrame(df_viz_pd)
df_viz.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_customer_pca")

print(f"   ml_customer_pca: {df_viz.count():,} records")
print(f"   Variance explained: PC1={pca_sk.explained_variance_ratio_[0]:.1%}, "
      f"PC2={pca_sk.explained_variance_ratio_[1]:.1%}")

# Cleanup
del X, X_scaled, X_pca, df_feat_pd, df_seg_pd
gc.collect()

display(df_viz.select("pc1", "pc2", "prediction", "total_spent"))

  PCA bằng scikit-learn (tránh Spark ML cache overflow)...
   ml_customer_pca: 250 records
   Variance explained: PC1=42.9%, PC2=16.9%


pc1,pc2,prediction,total_spent
-1.246946121089717,-0.3335299583236706,3,2000000.0
-0.8103769235639631,-1.0984396202661293,3,0.0
0.9482309821263575,2.285338497664892,2,7500000.0
-1.2568935996251112,-0.5882708689997852,3,0.0
1.395912660962788,0.16841900390773298,2,0.0
-1.1844395022720786,-0.7340976801099302,3,0.0
2.071520540350958,0.02144925411683425,0,6000000.0
2.1714207980983207,-0.906772368505632,0,0.0
1.4140261853010463,0.13196230113019677,2,0.0
1.5858445877963123,0.40436155223678083,2,4500000.0


### Cell 16 — Checklist kết thúc Day 6

In [0]:
# Chạy cell này để tự kiểm tra toàn bộ Day 6
import mlflow

spark.sql("USE tourgo")

print("=" * 60)
print("✅ CHECKLIST KẾT THÚC NGÀY 6")
print("=" * 60)

results = {}

# 1. customer_segments
try:
    n = spark.read.table("ml_customer_segments").count()
    k = spark.read.table("ml_customer_segments") \
        .select("prediction").distinct().count()
    results["K-Means model + customer_segments"] = \
        f"✅ {n:,} records, {k} cụm"
except:
    results["K-Means model + customer_segments"] = "❌ Chưa có"

# 2. Cluster profile
try:
    from pyspark.sql.functions import count, avg, round as _round
    profile = spark.read.table("ml_customer_segments") \
        .groupBy("prediction") \
        .agg(
            count("user_id").alias("size"),
            _round(avg("total_spent"), 0).alias("avg_spent"),
            _round(avg("cancellation_rate"), 3).alias("cancel_rate")
        ).orderBy("prediction")
    results["Cluster profile"] = f"✅ {profile.count()} cụm"
    profile.show(truncate=False)
except:
    results["Cluster profile"] = "❌ Lỗi"

# 3. PCA scatter data
try:
    n = spark.read.table("ml_customer_pca").count()
    results["PCA scatter data (ml_customer_pca)"] = f"✅ {n:,} records"
except:
    results["PCA scatter data (ml_customer_pca)"] = \
        "⚠️  Chưa có — chạy Cell 15 fix"

# 4. Tour recommendations
try:
    n = spark.read.table("ml_tour_recommendations").count()
    results["Tour recommendations"] = f"✅ {n} rows"
except:
    results["Tour recommendations"] = "❌ Chưa có"

# 5. MLflow runs
try:
    runs = mlflow.search_runs(
        experiment_names=["tourgo-ml-experiments"]
    )
    elbow_runs = runs[runs["tags.mlflow.runName"]
                      .str.contains("elbow", case=False, na=False)]
    final_runs = runs[runs["tags.mlflow.runName"]
                      .str.contains("final", case=False, na=False)]
    results["MLflow elbow run (K=2..8)"] = \
        f"✅ {len(elbow_runs)} run(s)"
    results["MLflow final model run"] = \
        f"✅ {len(final_runs)} run(s)"
except:
    results["MLflow runs"] = \
        "⚠️  Kiểm tra thủ công: sidebar → Experiments"

print(f"\n  {'Hạng mục':<40} | Kết quả")
print("  " + "-"*65)
for k, v in results.items():
    print(f"  {k:<40} | {v}")

print("\n📋 Checklist thủ công còn lại:")
print("  □ Chụp screenshot MLflow UI → docs/screenshots/day6/")
print("  □ [A] Hà đặt tên 4 cụm dựa trên cluster profile")
print("  □ [D] Khánh viết customer_segments_report.md")
print("  □ [C] Phụng làm scatter plot từ ml_customer_pca")

✅ CHECKLIST KẾT THÚC NGÀY 6
+----------+----+---------+-----------+
|prediction|size|avg_spent|cancel_rate|
+----------+----+---------+-----------+
|0         |57  |5522807.0|0.018      |
|1         |9   |0.0      |0.0        |
|2         |60  |1958333.0|0.117      |
|3         |124 |1763710.0|0.056      |
+----------+----+---------+-----------+



2026/06/02 11:27:30 WARNING mlflow.tracking.fluent: Cannot retrieve experiment by name tourgo-ml-experiments



  Hạng mục                                 | Kết quả
  -----------------------------------------------------------------
  K-Means model + customer_segments        | ✅ 250 records, 4 cụm
  Cluster profile                          | ✅ 4 cụm
  PCA scatter data (ml_customer_pca)       | ✅ 250 records
  Tour recommendations                     | ✅ 23 rows
  MLflow runs                              | ⚠️  Kiểm tra thủ công: sidebar → Experiments

📋 Checklist thủ công còn lại:
  □ Chụp screenshot MLflow UI → docs/screenshots/day6/
  □ [A] Hà đặt tên 4 cụm dựa trên cluster profile
  □ [D] Khánh viết customer_segments_report.md
  □ [C] Phụng làm scatter plot từ ml_customer_pca


### CELL CUỐI CÙNG: MÁP NHÃN PHÂN KHÚC KINH DOANH THEO QUYẾT ĐỊNH CỦA [A] HÀ

In [0]:
# ── CELL CUỐI CÙNG: MÁP NHÃN PHÂN KHÚC KINH DOANH THEO QUYẾT ĐỊNH CỦA [A] HÀ ──
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 1. SỬA LẠI ĐỂ KHỚP ĐÚNG SỐ LIỆU THỰC TẾ TRÊN MÀN HÌNH CỦA BẠN:
cluster_names = {
    0: "🏆 Khách VIP",             # Chi tiêu cao nhất (5.52 triệu), tỷ lệ hủy cực thấp (1.8%)
    1: "💤 Khách Ngủ Đông",        # Chi tiêu trung bình (1.95 triệu), tỷ lệ hủy khá cao (11.7%)
    2: "⚠️ Khách Không Ổn Định",   # Chi tiêu thấp hoặc trung bình, cần chú ý nhãn này dựa trên profile gốc
    3: "🌱 Khách Tiềm Năng"        # Chi tiêu = 0, hủy = 0 (Khách mới tinh, hệ thống cần kích cầu)
}

# 2. Tạo hàm UDF để ánh xạ từ số cụm sang tên phân khúc
map_cluster_name_udf = udf(lambda pred: cluster_names.get(pred, "Chưa phân loại"), StringType())

# 3. Thêm cột 'segment_name' trực quan vào DataFrame kết quả
df_final_segmented = df_clustered.withColumn("segment_name", map_cluster_name_udf("prediction"))

# 4. Hiển thị bảng tổng hợp có nhãn chữ tiếng Việt chuẩn xác
print("📊 BẢNG TỔNG HỢP PHÂN KHÚC KHÁCH HÀNG TOURGO CHÍNH THỨC (Đã sửa đổi):")
display(
    df_final_segmented.groupBy("segment_name")
    .agg(
        count("user_id").alias("Số lượng khách"),
        _round(avg("total_spent"), 0).alias("Chi tiêu trung bình"),
        _round(avg("cancellation_rate") * 100, 1).alias("Tỷ lệ hủy đơn (%)")
    )
    .orderBy("Chi tiêu trung bình", ascending=False)
)

📊 BẢNG TỔNG HỢP PHÂN KHÚC KHÁCH HÀNG TOURGO CHÍNH THỨC (Đã sửa đổi):


segment_name,Số lượng khách,Chi tiêu trung bình,Tỷ lệ hủy đơn (%)
🏆 Khách VIP,57,5522807.0,1.8
⚠️ Khách Không Ổn Định,60,1958333.0,11.7
🌱 Khách Tiềm Năng,124,1763710.0,5.6
💤 Khách Ngủ Đông,9,0.0,0.0
